In [ ]:
from pathlib import Path
import json

nb_path = Path("/mnt/data/iteration4_bdas.ipynb")

cells = []

def add_md(text):
    cells.append({
        "cell_type": "markdown",
        "metadata": {},
        "source": text.splitlines(True)
    })

def add_code(text):
    cells.append({
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": text.splitlines(True)
    })

add_md("""# INFOSYS 722 Iteration 4 - BDAS
## Diabetes Prediction using AWS EC2 + PySpark + Spark MLlib

Technology stack:
- AWS EC2
- Apache Spark
- PySpark
- Spark MLlib

KDD Steps:
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Data Transformation
5. Data Mining Method
6. Data Mining Algorithm
7. Data Mining
8. Interpretation
""")

add_code("""from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = (
    SparkSession.builder
    .appName("INFOSYS722-BDAS-Diabetes")
    .getOrCreate()
)

print("Spark Version:", spark.version)
""")

add_md("""## Step 1 - Business Understanding

Objective:
Develop a scalable diabetes risk prediction solution using Big Data Analytics technologies.

The goal is to reproduce the OSAS machine learning pipeline using PySpark/Spark MLlib.
""")

add_md("""## Step 2 - Data Understanding""")

add_code("""DATA_PATH = "data/brfss2023_diabetes_analysis.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH)
)

print("Records:", df.count())

df.printSchema()

df.describe().show()
""")

add_md("""## Step 3 - Data Preparation""")

add_code("""target = "Diabetes_binary"

print("Missing values before cleaning")

for c in df.columns:
    missing = df.filter(col(c).isNull()).count()
    if missing > 0:
        print(c, missing)

df_clean = df.dropna(subset=[target])

df_clean = df_clean.fillna(0)

print("Records after cleaning:", df_clean.count())
""")

add_md("""## Step 4 - Data Transformation

Spark MLlib requires input variables to be assembled into a feature vector.
""")

add_code("""features = [
    c for c in df_clean.columns
    if c != target
]

assembler = VectorAssembler(
    inputCols=features,
    outputCol="features"
)

df_model = assembler.transform(df_clean)

df_model = df_model.select(
    "features",
    target
)

df_model.show(5)
""")

add_md("""## Step 5 - Data Mining Method

Problem type:
Binary Classification

Algorithms:
- Logistic Regression
- Random Forest
""")

add_md("""## Step 6 & 7 - Algorithm Selection and Data Mining""")

add_code("""df_model = df_model.withColumnRenamed(
    target,
    "label"
)

train, test = df_model.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training:", train.count())
print("Testing:", test.count())
""")

add_code("""lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

lr_model = lr.fit(train)

lr_prediction = lr_model.transform(test)

lr_prediction.select(
    "label",
    "prediction",
    "probability"
).show(5)
""")

add_code("""rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

rf_model = rf.fit(train)

rf_prediction = rf_model.transform(test)

rf_prediction.select(
    "label",
    "prediction",
    "probability"
).show(5)
""")

add_md("""## Step 8 - Interpretation and Evaluation""")

add_code("""auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

auc = auc_evaluator.evaluate(rf_prediction)
accuracy = accuracy_evaluator.evaluate(rf_prediction)

print("Random Forest AUC:", auc)
print("Random Forest Accuracy:", accuracy)
""")

add_code("""rf_prediction.groupBy(
    "label",
    "prediction"
).count().show()
""")

add_md("""## Save Results

The prediction output can be exported for report evidence and visualisation.
""")

add_code("""(
    rf_prediction
    .select("label", "prediction", "probability")
    .write
    .mode("overwrite")
    .csv("../output/rf_prediction")
)
""")

add_code("""spark.stop()
""")

notebook = {
    "cells": cells,
    "metadata": {
        "kernelspec": {
            "display_name": "PySpark",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

nb_path.write_text(json.dumps(notebook, indent=2), encoding="utf-8")

str(nb_path)
